# Notebook 02 — Model Training Analysis

**Mục đích:** Phân tích chi tiết kết quả training model Late Fusion (XLM-R + FT-Transformer).  
**Yêu cầu:** Chạy sau khi `src/train.py` và `scripts/run_ablation.py` đã hoàn thành.

---

## Nội dung
1. Load & kiểm tra results files
2. Learning curves (train loss vs val loss)
3. Confusion matrix (test set)
4. Per-class F1 bar chart
5. Ablation study results table
6. Error analysis — top sai nhiều nhất mỗi lớp
7. LIME explanation examples

In [ ]:
import sys, json
from pathlib import Path

PROJECT_ROOT = Path("..")
sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from sklearn.metrics import (
    confusion_matrix, classification_report, f1_score
)

sns.set_theme(style="whitegrid", font_scale=1.1)
plt.rcParams["figure.dpi"] = 120

FIGURES_DIR = PROJECT_ROOT / "reports" / "figures"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

CLASS_NAMES = ["joy", "sadness", "anger", "fear", "disgust", "surprise", "neutral"]
PALETTE     = sns.color_palette("husl", len(CLASS_NAMES))

print("Setup OK")

---
## 1. Load Results Files

In [ ]:
# ── Training metrics log (written by src/train.py) ────────────────────────
METRICS_LOG = PROJECT_ROOT / "models" / "best_model" / "training_metrics.json"

if METRICS_LOG.exists():
    with open(METRICS_LOG) as f:
        train_history = json.load(f)  # List[{epoch, train_loss, val_loss, val_f1}]
    history_df = pd.DataFrame(train_history)
    print(f"Training history: {len(history_df)} epochs")
    print(history_df.tail())
else:
    print("⚠️  training_metrics.json not found — run src/train.py first")
    # Synthetic placeholder for notebook development
    history_df = pd.DataFrame({
        "epoch":      list(range(1, 11)),
        "train_loss": [1.95, 1.62, 1.41, 1.24, 1.11, 1.01, 0.93, 0.87, 0.83, 0.80],
        "val_loss":   [1.80, 1.55, 1.38, 1.26, 1.18, 1.14, 1.13, 1.14, 1.17, 1.19],
        "val_f1":     [0.30, 0.40, 0.50, 0.57, 0.61, 0.64, 0.65, 0.65, 0.64, 0.63],
    })
    print("Using SYNTHETIC placeholder data — replace with real results after training")

In [ ]:
# ── Test set predictions (written by src/evaluate.py) ────────────────────
PREDS_PATH = PROJECT_ROOT / "models" / "best_model" / "test_predictions.csv"

if PREDS_PATH.exists():
    preds_df = pd.read_csv(PREDS_PATH)   # columns: text, true_label, pred_label
    print(f"Test predictions: {len(preds_df)} samples")
    print(preds_df.head(3))
else:
    print("⚠️  test_predictions.csv not found — run src/evaluate.py first")
    # Synthetic placeholder
    rng = np.random.default_rng(42)
    n_test = 266
    true_labels = rng.choice(CLASS_NAMES, n_test,
                             p=[0.30, 0.12, 0.14, 0.11, 0.10, 0.11, 0.12])
    # Simulate ~65% accuracy with confusion
    pred_labels = true_labels.copy()
    noise_idx = rng.choice(n_test, int(n_test * 0.35), replace=False)
    pred_labels[noise_idx] = rng.choice(CLASS_NAMES, len(noise_idx))
    preds_df = pd.DataFrame({
        "text":       [f"sample_{i}" for i in range(n_test)],
        "true_label": true_labels,
        "pred_label": pred_labels,
    })
    print("Using SYNTHETIC placeholder predictions")

In [ ]:
# ── Ablation results ─────────────────────────────────────────────────────
ABLATION_PATH = PROJECT_ROOT / "reports" / "ablation_results.csv"

if ABLATION_PATH.exists():
    ablation_df = pd.read_csv(ABLATION_PATH)
    print("Ablation results loaded:")
    print(ablation_df)
else:
    print("⚠️  ablation_results.csv not found — run scripts/run_ablation.py first")
    # Synthetic placeholder
    ablation_df = pd.DataFrame({
        "experiment": [
            "Exp1: XLM-R only",
            "Exp2: XLM-R + Teencode",
            "Exp3: Full Fusion (XLM-R + Teencode + FT-Transformer)",
        ],
        "f1_macro":  [0.620, 0.657, 0.701],
        "accuracy":  [0.672, 0.705, 0.738],
        "precision": [0.631, 0.665, 0.712],
        "recall":    [0.614, 0.651, 0.692],
    })
    print("Using SYNTHETIC placeholder ablation data")

---
## 2. Learning Curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ── Loss curve ───────────────────────────────────────────────────────────
ax = axes[0]
ax.plot(history_df["epoch"], history_df["train_loss"],
        marker="o", label="Train Loss", color="#2196F3", linewidth=2)
ax.plot(history_df["epoch"], history_df["val_loss"],
        marker="s", label="Val Loss",   color="#FF5722", linewidth=2, linestyle="--")

best_epoch = history_df["val_loss"].idxmin()
ax.axvline(x=history_df.loc[best_epoch, "epoch"], color="gray",
           linestyle=":", alpha=0.7, label=f"Best epoch ({history_df.loc[best_epoch,'epoch']})")

ax.set_xlabel("Epoch")
ax.set_ylabel("Cross-Entropy Loss")
ax.set_title("Training & Validation Loss")
ax.legend()
ax.grid(True, alpha=0.4)

# ── F1 curve ─────────────────────────────────────────────────────────────
ax = axes[1]
ax.plot(history_df["epoch"], history_df["val_f1"],
        marker="D", color="#4CAF50", linewidth=2, label="Val F1-Macro")

best_f1_epoch = history_df["val_f1"].idxmax()
best_f1 = history_df.loc[best_f1_epoch, "val_f1"]
ax.axvline(x=history_df.loc[best_f1_epoch, "epoch"], color="gray",
           linestyle=":", alpha=0.7)
ax.annotate(f"Best: {best_f1:.3f}",
            xy=(history_df.loc[best_f1_epoch, "epoch"], best_f1),
            xytext=(history_df.loc[best_f1_epoch, "epoch"] + 0.5, best_f1 - 0.03),
            fontsize=10, color="#4CAF50")

ax.set_xlabel("Epoch")
ax.set_ylabel("F1-Macro")
ax.set_title("Validation F1-Macro per Epoch")
ax.legend()
ax.grid(True, alpha=0.4)

plt.tight_layout()
plt.savefig(FIGURES_DIR / "learning_curves.png", bbox_inches="tight")
plt.show()
print(f"Best val_loss = {history_df['val_loss'].min():.4f} (epoch {history_df.loc[best_epoch,'epoch']})")
print(f"Best val F1   = {best_f1:.4f} (epoch {history_df.loc[best_f1_epoch,'epoch']})")

---
## 3. Confusion Matrix

In [ ]:
y_true = preds_df["true_label"].tolist()
y_pred = preds_df["pred_label"].tolist()

cm = confusion_matrix(y_true, y_pred, labels=CLASS_NAMES)
cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)  # row-normalize

fig, axes = plt.subplots(1, 2, figsize=(18, 7))

for ax, matrix, fmt, title, vmax in [
    (axes[0], cm,      "d",   "Confusion Matrix (counts)",    None),
    (axes[1], cm_norm, ".2f", "Confusion Matrix (normalized)", 1.0),
]:
    sns.heatmap(
        matrix, annot=True, fmt=fmt, cmap="Blues",
        xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES,
        linewidths=0.5, vmax=vmax, ax=ax,
    )
    ax.set_xlabel("Predicted", fontsize=12)
    ax.set_ylabel("True", fontsize=12)
    ax.set_title(title, fontsize=13, fontweight="bold")
    ax.tick_params(axis="x", rotation=30)
    ax.tick_params(axis="y", rotation=0)

plt.tight_layout()
plt.savefig(FIGURES_DIR / "confusion_matrix.png", bbox_inches="tight")
plt.show()

f1_macro = f1_score(y_true, y_pred, labels=CLASS_NAMES, average="macro", zero_division=0)
print(f"\nTest F1-Macro: {f1_macro:.4f}")

---
## 4. Per-Class F1 Bar Chart

In [ ]:
report = classification_report(
    y_true, y_pred, labels=CLASS_NAMES,
    output_dict=True, zero_division=0
)

per_class = pd.DataFrame(
    {cls: report[cls] for cls in CLASS_NAMES if cls in report}
).T.reset_index().rename(columns={"index": "emotion"})

fig, axes = plt.subplots(1, 3, figsize=(18, 5), sharey=True)

for ax, metric in zip(axes, ["precision", "recall", "f1-score"]):
    colors = [
        "#4CAF50" if v >= 0.70 else "#FFC107" if v >= 0.55 else "#F44336"
        for v in per_class[metric]
    ]
    bars = ax.barh(per_class["emotion"], per_class[metric], color=colors, edgecolor="white")
    ax.axvline(x=report["macro avg"][metric], color="navy", linestyle="--",
               linewidth=1.5, label=f"Macro avg = {report['macro avg'][metric]:.3f}")
    for bar, val in zip(bars, per_class[metric]):
        ax.text(val + 0.005, bar.get_y() + bar.get_height() / 2,
                f"{val:.3f}", va="center", fontsize=9)
    ax.set_xlim(0, 1.05)
    ax.set_title(metric.capitalize(), fontsize=12, fontweight="bold")
    ax.set_xlabel("Score")
    ax.legend(fontsize=9)
    ax.grid(axis="x", alpha=0.3)

legend_patches = [
    mpatches.Patch(color="#4CAF50", label="≥ 0.70 (Good)"),
    mpatches.Patch(color="#FFC107", label="0.55–0.70 (Fair)"),
    mpatches.Patch(color="#F44336", label="< 0.55 (Needs work)"),
]
fig.legend(handles=legend_patches, loc="lower center", ncol=3,
           bbox_to_anchor=(0.5, -0.08), fontsize=10)

plt.suptitle("Per-Class Precision / Recall / F1 on Test Set",
             fontsize=14, fontweight="bold", y=1.02)
plt.tight_layout()
plt.savefig(FIGURES_DIR / "per_class_metrics.png", bbox_inches="tight")
plt.show()

print("\n" + classification_report(y_true, y_pred, labels=CLASS_NAMES, zero_division=0))

---
## 5. Ablation Study Results

In [ ]:
fig, ax = plt.subplots(figsize=(11, 5))

x = np.arange(len(ablation_df))
width = 0.2
metrics = ["f1_macro", "accuracy", "precision", "recall"]
colors  = ["#2196F3", "#4CAF50", "#FF9800", "#E91E63"]
labels  = ["F1-Macro", "Accuracy", "Precision", "Recall"]

for i, (metric, color, label) in enumerate(zip(metrics, colors, labels)):
    bars = ax.bar(x + i * width, ablation_df[metric], width,
                  label=label, color=color, alpha=0.85, edgecolor="white")
    for bar, val in zip(bars, ablation_df[metric]):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.005,
                f"{val:.3f}", ha="center", va="bottom", fontsize=8)

ax.set_xticks(x + width * 1.5)
ax.set_xticklabels(ablation_df["experiment"], rotation=12, ha="right", fontsize=9)
ax.set_ylim(0, 0.85)
ax.set_ylabel("Score")
ax.set_title("Ablation Study: Contribution of Each Component",
             fontsize=13, fontweight="bold")
ax.legend(loc="upper left", fontsize=9)
ax.grid(axis="y", alpha=0.3)

# Draw delta annotations for F1-Macro gains
f1_vals = ablation_df["f1_macro"].tolist()
for i in range(1, len(f1_vals)):
    delta = f1_vals[i] - f1_vals[i - 1]
    ax.annotate(
        f"+{delta:.3f}",
        xy=(x[i], f1_vals[i] + 0.025),
        ha="center", fontsize=9, color="#2196F3", fontweight="bold"
    )

plt.tight_layout()
plt.savefig(FIGURES_DIR / "ablation_results.png", bbox_inches="tight")
plt.show()

print("\nAblation Table:")
print(ablation_df.to_string(index=False))

---
## 6. Error Analysis — Sai nhiều nhất theo lớp

In [ ]:
errors_df = preds_df[preds_df["true_label"] != preds_df["pred_label"]].copy()
errors_df["mistake"] = errors_df["true_label"] + " → " + errors_df["pred_label"]

print(f"Total errors: {len(errors_df)} / {len(preds_df)} ({len(errors_df)/len(preds_df)*100:.1f}%)")
print("\nMost common confusion pairs:")
print(errors_df["mistake"].value_counts().head(10).to_string())

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(22, 10))
axes = axes.flatten()

for ax, cls in zip(axes, CLASS_NAMES):
    cls_errors = errors_df[errors_df["true_label"] == cls]
    if len(cls_errors) == 0:
        ax.text(0.5, 0.5, f"{cls}\n(no errors)",
                ha="center", va="center", transform=ax.transAxes, fontsize=11)
        ax.axis("off")
        continue

    mistake_counts = cls_errors["pred_label"].value_counts()
    colors_bar = [PALETTE[CLASS_NAMES.index(lbl)] for lbl in mistake_counts.index]
    bars = ax.bar(mistake_counts.index, mistake_counts.values,
                  color=colors_bar, edgecolor="white")
    for bar, val in zip(bars, mistake_counts.values):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.1,
                str(val), ha="center", fontsize=9)
    ax.set_title(f"True: {cls} (n_err={len(cls_errors)})", fontweight="bold")
    ax.set_xlabel("Predicted as")
    ax.set_ylabel("Count")
    ax.tick_params(axis="x", rotation=30)

axes[-1].axis("off")  # hide last empty subplot

plt.suptitle("Error Distribution: What does the model confuse each class with?",
             fontsize=14, fontweight="bold", y=1.01)
plt.tight_layout()
plt.savefig(FIGURES_DIR / "error_analysis.png", bbox_inches="tight")
plt.show()

In [ ]:
# Show 3 most-confused examples per class
for cls in CLASS_NAMES:
    cls_errors = errors_df[errors_df["true_label"] == cls].head(3)
    if len(cls_errors) == 0:
        continue
    print(f"\n{'='*60}")
    print(f"True: {cls.upper()} — Top errors")
    print(f"{'='*60}")
    for _, row in cls_errors.iterrows():
        preview = str(row.get('text', ''))[:100]
        print(f"  Text:      {preview}")
        print(f"  Predicted: {row['pred_label']}")
        print()

---
## 7. LIME Explanation Examples

Chạy cell này sau khi model checkpoint đã được load thành công.

In [ ]:
from IPython.display import HTML, display

CHECKPOINT = PROJECT_ROOT / "models" / "best_model"

if not (CHECKPOINT / "config.json").exists():
    print("⚠️  Model checkpoint not found — skipping LIME demo.")
    print(f"   Expected at: {CHECKPOINT}")
else:
    from app.inference import LateFusionPredictor
    from app.explainer import TextExplainer

    predictor = LateFusionPredictor(str(CHECKPOINT))
    explainer = TextExplainer(predictor)

    sample_texts = [
        "hôm nay vui quá luôn 😊 mọi việc đều suôn sẻ!",
        "tức vcl, sao làm ăn kiểu này, chán lắm rồi",
        "không biết gì luôn, sợ quá 😱",
        "đồ ăn dở tệ, ăn không được luôn",
        "ủa sao lại vậy được? bất ngờ thật!",
    ]

    for text in sample_texts:
        result = explainer.explain(text)
        print(f"\nText: {text}")
        print(f"Prediction: {result.label} ({result.confidence:.3f})")
        display(HTML(result.highlight_html))

---
## 8. Summary — Bảng kết quả tổng hợp

In [ ]:
summary = pd.DataFrame([
    {
        "Model": row["experiment"],
        "F1-Macro": f"{row['f1_macro']:.3f}",
        "Accuracy": f"{row['accuracy']:.3f}",
        "Precision": f"{row['precision']:.3f}",
        "Recall": f"{row['recall']:.3f}",
    }
    for _, row in ablation_df.iterrows()
])

print("=" * 90)
print("FINAL RESULTS TABLE")
print("=" * 90)
print(summary.to_string(index=False))
print("=" * 90)

best_exp = ablation_df.loc[ablation_df["f1_macro"].idxmax()]
print(f"\n✅ Best model: {best_exp['experiment']}")
print(f"   F1-Macro = {best_exp['f1_macro']:.3f}")
print(f"   Accuracy = {best_exp['accuracy']:.3f}")